# 02 · Experiments

3 representations × 3 architectures, each evaluated with k-fold cross-validation.
Every run saves its fold scores to `results/<architecture>__<representation>.json`.

In [40]:
# Setup: make the course code (part1) and our code (src) importable.
# Run notebooks from this folder (JupyterLab / VS Code do this by default).
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
from part1 import experiment
from src import config, cv, data, encoders, results_io

## Data and folds

In [42]:
LABEL_MODE = "fine"   # "fine" (28) | "ekman" (7) | "sentiment" (4)
pairs, outcome_space, stats = data.load_pairs(label_mode=LABEL_MODE)
print(stats)
folds = experiment.cross_validation_split(pairs, k=config.K_FOLDS, seed=config.SEED)

{'label_mode': 'fine', 'min_raters': 2, 'comments_with_label': 54263, 'kept': 45446, 'dropped_multilabel': 8817}


## Tokeniser

Reuse the tutorial's or train our own? On what corpus, which vocab size?
Note the choice + reason in the decisions log.

In [43]:
corpus = data.tokenizer_corpus()
tokenizer = experiment.train_tokenizer(corpus, vocab_size=58800, min_frequency=1)
tokenizer.get_vocab_size()

58800

## Representations

In [44]:
representations = {
    "indicator": encoders.TokenIndicator(tokenizer),
    "bow": encoders.BagOfWords(tokenizer),
    # Third: TF-IDF. IDF weights are fitted per fold on its training texts only (no leakage),
    # so this is a {fold: encoder} dict; the CV functions below accept both forms.
    "tfidf": encoders.fit_per_fold(folds, lambda: encoders.TfIdf(tokenizer)),
}

## Log-linear

In [45]:
for rep_name, enc in representations.items():
    name = f"loglin__{rep_name}"
    out = cv.course_cv(folds, outcome_space, enc, "logistic", save_to=str(config.MODELS_DIR / name))
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size())
    print(name, out["accuracies"])

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: T

loglin__indicator OrderedDict({0: 0.5381738173817382, 1: 0.5538563098250633, 2: 0.5384530751457807, 3: 0.5441742765980856, 4: 0.5376829134118165})


  0%|          | 0/5 [00:00<?, ?it/s]

/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: T

loglin__bow OrderedDict({0: 0.5377337733773377, 1: 0.5511057322037628, 2: 0.534712289580812, 3: 0.5427439762350094, 4: 0.5393332599845968})


... (8 outputs left out)

/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  0%|          | 0/1 [00:00<?, ?it/s]

/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


loglin__tfidf OrderedDict({0: 0.5115511551155115, 1: 0.5240400484101662, 2: 0.5073165364726593, 3: 0.511827483771592, 4: 0.5090769061502916})


## MLP (one hidden layer)

In [46]:
smlp_args = dict(hidden_dim=128, dropout=0.1, num_epochs=10)  # TODO: tune

for rep_name, enc in representations.items():
    name = f"mlp__{rep_name}"
    out = cv.course_cv(folds, outcome_space, enc, "mlp", save_to=str(config.MODELS_DIR / name),
                       smlp_args=smlp_args)
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size(), **smlp_args)
    print(name, out["accuracies"])

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 1.638 seconds
Dataloader done! Time taken: 8.275 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 2.2700 | train acc: 0.4277 | val loss: 1.8488 | val acc: 0.5089
Epoch 02 | train loss: 1.2995 | train acc: 0.6262 | val loss: 1.6874 | val acc: 0.5425
Epoch 03 | train loss: 0.6943 | train acc: 0.8027 | val loss: 1.7670 | val acc: 0.5321
Epoch 04 | train loss: 0.3823 | train acc: 0.8989 | val loss: 1.9285 | val acc: 0.5182
Epoch 05 | train loss: 0.2313 | train acc: 0.9425 | val loss: 2.1015 | val acc: 0.5138
Epoch 06 | train loss: 0.1515 | train acc: 0.9641 | val loss: 2.3211 | val acc: 0.5087
Epoch 07 | train loss: 0.1045 | train acc: 0.9763 | val loss: 2.5122 | val acc: 0.5031
Epoch 08 | train loss: 0.0765 | train acc: 0.9828 | val loss: 2.7055 | val acc: 0.4992
Epoch 09 | train loss: 0.0585 | train acc: 0.9867 | val loss: 2.8953 | val acc: 0.4937
Epoch 10 | train loss: 0.0453 | train acc: 0.9903 | val loss: 3.0452 | val acc: 0.4884
Training done! Time taken: 34.191 seconds
Transforming documents...
Document transformation done! Time taken: 1.625 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 2.2630 | train acc: 0.4333 | val loss: 1.8096 | val acc: 0.5280
Epoch 02 | train loss: 1.3062 | train acc: 0.6249 | val loss: 1.6455 | val acc: 0.5558
Epoch 03 | train loss: 0.7027 | train acc: 0.8019 | val loss: 1.7156 | val acc: 0.5455
Epoch 04 | train loss: 0.3858 | train acc: 0.8974 | val loss: 1.8944 | val acc: 0.5375
Epoch 05 | train loss: 0.2346 | train acc: 0.9406 | val loss: 2.0396 | val acc: 0.5235
Epoch 06 | train loss: 0.1544 | train acc: 0.9636 | val loss: 2.2198 | val acc: 0.5155
Epoch 07 | train loss: 0.1084 | train acc: 0.9755 | val loss: 2.3625 | val acc: 0.5091
Epoch 08 | train loss: 0.0791 | train acc: 0.9823 | val loss: 2.5604 | val acc: 0.5108
Epoch 09 | train loss: 0.0599 | train acc: 0.9865 | val loss: 2.7467 | val acc: 0.5029
Epoch 10 | train loss: 0.0477 | train acc: 0.9894 | val loss: 2.9065 | val acc: 0.4982
Training done! Time taken: 34.240 seconds
Transforming documents...
Document transformation done! Time taken: 1.600 seconds
Datalo

... (35 outputs left out)

  0%|          | 0/1 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 4.248 seconds
Dataloader done! Time taken: 7.851 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 2.4809 | train acc: 0.3774 | val loss: 2.1137 | val acc: 0.4545
Epoch 02 | train loss: 1.6977 | train acc: 0.5353 | val loss: 1.8118 | val acc: 0.5196
Epoch 03 | train loss: 1.0863 | train acc: 0.6988 | val loss: 1.7553 | val acc: 0.5283
Epoch 04 | train loss: 0.6594 | train acc: 0.8333 | val loss: 1.8126 | val acc: 0.5096
Epoch 05 | train loss: 0.4052 | train acc: 0.9050 | val loss: 1.9151 | val acc: 0.4990
Epoch 06 | train loss: 0.2616 | train acc: 0.9427 | val loss: 2.0356 | val acc: 0.4906
Epoch 07 | train loss: 0.1773 | train acc: 0.9628 | val loss: 2.1750 | val acc: 0.4861
Epoch 08 | train loss: 0.1250 | train acc: 0.9744 | val loss: 2.3116 | val acc: 0.4783
Epoch 09 | train loss: 0.0899 | train acc: 0.9829 | val loss: 2.4242 | val acc: 0.4741
Epoch 10 | train loss: 0.0672 | train acc: 0.9876 | val loss: 2.6027 | val acc: 0.4784
Training done! Time taken: 34.490 seconds
mlp__tfidf OrderedDict({0: 0.5298129812981298, 1: 0.5443943228077897, 2: 0.5296512267576

## Our third architecture: deep low-rank MLP

`src/models.py::DeepLowRankMLP`: several hidden layers, each weight matrix factorised as W = A·B with rank r
(the LoRA factorisation, trained from scratch: there is no pretrained model to adapt).
The first layer (vocab × hidden) holds most parameters; the rank bottleneck shrinks it so we can stack more layers.
Residual connections between equal-width layers keep the deep network trainable.

In [47]:
import numpy as np
import pandas as pd
from src import models

# Quick check before the full CV run: parameter counts of full-rank vs low-rank
V, C = tokenizer.get_vocab_size(), len(outcome_space)
for r in [None, 64, 16]:
    m = models.DeepLowRankMLP(V, [256, 256, 256, 256], C, rank=r)
    print(f"rank={r}: {models.count_parameters(m):,} parameters")

rank=None: 15,257,628 parameters
rank=64: 3,886,108 parameters
rank=16: 977,692 parameters


In [48]:
deep_args = dict(hidden_dims=[256, 256, 256, 256], rank=32, dropout=0.5, residual=True)  # TODO: tune
# No early stopping: every fold trains all epochs, so the table shows where it starts to overfit.
# The reported CV accuracy is still that of the checkpoint with the lowest val loss (as in part1.smlp).
deep_train_args = dict(num_epochs=10, learning_rate=1e-3, batch_size=128, seed=config.SEED,
                       device="auto")   # "auto" = CUDA if available, else CPU (also on a Mac)
N_JOBS = None   # parallel training processes; None = half the cores on CPU, 1 on a GPU; 1 = sequential

outs = models.run_experiments(
    {f"deeplowrank__{rep}": dict(encoder=enc, model_args=deep_args) for rep, enc in representations.items()},
    folds, outcome_space, save_root=config.MODELS_DIR, train_args=deep_train_args, n_jobs=N_JOBS,
)
for name, out in outs.items():
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size(), **deep_args, **deep_train_args)
    acc = list(out["accuracies"].values())
    print(f"{name}: CV accuracy {np.mean(acc):.4f} ± {np.std(acc):.4f}  (mean ± std over {len(acc)} folds)")
    display(models.epoch_table(out))

encoding:   0%|          | 0/7 [00:00<?, ?it/s]

3 experiments x 5 folds:   0%|          | 0/15 [00:00<?, ?it/s]

deeplowrank__indicator: CV accuracy 0.5032 ± 0.0083  (mean ± std over 5 folds)


,train loss,train accuracy,val loss,val accuracy
epoch,,,,
1,2.372 ± 0.006,0.409 ± 0.002,2.046 ± 0.026,0.480 ± 0.008
2 *,1.730 ± 0.011,0.531 ± 0.002,1.964 ± 0.012,0.503 ± 0.008
3,1.218 ± 0.014,0.647 ± 0.003,2.095 ± 0.024,0.484 ± 0.007
4,0.845 ± 0.010,0.750 ± 0.002,2.416 ± 0.054,0.466 ± 0.006
5,0.603 ± 0.006,0.824 ± 0.002,2.744 ± 0.088,0.457 ± 0.002
6,0.447 ± 0.009,0.869 ± 0.001,3.053 ± 0.107,0.445 ± 0.005
7,0.349 ± 0.008,0.899 ± 0.003,3.482 ± 0.070,0.447 ± 0.009
8,0.285 ± 0.005,0.916 ± 0.002,3.660 ± 0.103,0.436 ± 0.005
9,0.246 ± 0.006,0.927 ± 0.001,3.928 ± 0.070,0.438 ± 0.005


deeplowrank__bow: CV accuracy 0.5007 ± 0.0069  (mean ± std over 5 folds)


,train loss,train accuracy,val loss,val accuracy
epoch,,,,
1,2.374 ± 0.006,0.409 ± 0.002,2.054 ± 0.024,0.480 ± 0.007
2 *,1.735 ± 0.011,0.531 ± 0.002,1.963 ± 0.012,0.501 ± 0.007
3,1.217 ± 0.010,0.648 ± 0.003,2.098 ± 0.016,0.488 ± 0.009
4,0.840 ± 0.011,0.753 ± 0.004,2.440 ± 0.071,0.467 ± 0.003
5,0.601 ± 0.012,0.825 ± 0.004,2.728 ± 0.064,0.457 ± 0.005
6,0.447 ± 0.014,0.869 ± 0.004,3.070 ± 0.084,0.445 ± 0.005
7,0.350 ± 0.015,0.899 ± 0.004,3.492 ± 0.084,0.444 ± 0.009
8,0.284 ± 0.010,0.916 ± 0.003,3.616 ± 0.095,0.430 ± 0.006
9,0.244 ± 0.011,0.929 ± 0.004,3.948 ± 0.081,0.438 ± 0.004


deeplowrank__tfidf: CV accuracy 0.4800 ± 0.0078  (mean ± std over 5 folds)


,train loss,train accuracy,val loss,val accuracy
epoch,,,,
1,2.457 ± 0.004,0.384 ± 0.001,2.171 ± 0.028,0.442 ± 0.009
2 *,1.864 ± 0.016,0.493 ± 0.005,2.087 ± 0.019,0.480 ± 0.008
3,1.388 ± 0.019,0.592 ± 0.005,2.237 ± 0.026,0.452 ± 0.015
4,1.040 ± 0.021,0.684 ± 0.005,2.574 ± 0.087,0.429 ± 0.008
5,0.784 ± 0.019,0.763 ± 0.006,2.878 ± 0.046,0.411 ± 0.007
6,0.595 ± 0.013,0.822 ± 0.005,3.293 ± 0.096,0.402 ± 0.012
7,0.462 ± 0.013,0.865 ± 0.004,3.697 ± 0.131,0.397 ± 0.004
8,0.367 ± 0.009,0.893 ± 0.003,4.104 ± 0.172,0.395 ± 0.015
9,0.306 ± 0.006,0.911 ± 0.002,4.385 ± 0.194,0.395 ± 0.009


### Rank sweep

Same architecture and training, one CV run per rank r (and full rank) for every representation.
All (rank × representation × fold) trainings share one process pool.
Results go to `results/sweeps/rank/` so they stay out of the main comparison in 03_analysis;
reload one later with `results_io.load(name, subdir="sweeps/rank")` and pass it to `models.epoch_table`.

In [49]:
RANKS = [4, 8, 16, 32, 64, 128, None]   # None = full rank (plain deep MLP)
SWEEP_DIR = "sweeps/rank"

sweep = {f"deeplowrank-r{r or 'full'}__{rep}": dict(encoder=enc, model_args={**deep_args, "rank": r})
         for rep, enc in representations.items() for r in RANKS}
sweep_outs = models.run_experiments(
    sweep, folds, outcome_space, save_root=config.MODELS_DIR / SWEEP_DIR,
    train_args=deep_train_args, n_jobs=N_JOBS,
)

rows = []
for name, out in sweep_outs.items():
    cfg = sweep[name]["model_args"]
    results_io.save_cv(name, out, subdir=SWEEP_DIR, vocab_size=tokenizer.get_vocab_size(), **cfg, **deep_train_args)
    rows.append(models.summary_row(out, representation=name.split("__")[1], rank=cfg["rank"] or "full"))
pd.DataFrame(rows).set_index(["representation", "rank"])

encoding:   0%|          | 0/7 [00:00<?, ?it/s]

21 experiments x 5 folds:   0%|          | 0/105 [00:00<?, ?it/s]

params cv acc (best ckpt) best epoch  \
representation rank                                           
indicator      4       250588    0.4767 ± 0.0063  2 2 2 2 2   
               8       492956    0.4940 ± 0.0064  2 2 2 2 2   
               16      977692    0.5034 ± 0.0056  2 2 2 2 2   
               32     1947164    0.5032 ± 0.0083  2 2 2 2 2   
               64     3886108    0.5135 ± 0.0051  2 2 2 2 2   
               128    7763996    0.5153 ± 0.0061  2 2 2 2 2   
               full  15257628    0.5299 ± 0.0046  2 2 2 2 2   
bow            4       250588    0.4759 ± 0.0066  2 2 2 2 2   
               8       492956    0.4924 ± 0.0061  2 2 2 2 2   
               16      977692    0.5027 ± 0.0056  2 2 2 2 2   
               32     1947164    0.5007 ± 0.0069  2 2 2 2 2   
               64     3886108    0.5113 ± 0.0045  2 2 2 2 2   
               128    7763996    0.5152 ± 0.0042  2 2 2 2 2   
               full  15257628    0.5282 ± 0.0038  2 2 2 2 2   
tfidf          4       250588    0.4542 ± 0.0146  2 2 1 2 2   
               8       492956    0.4695 ± 0.0096  2 2 2 2 2   
               16      977692    0.4775 ± 0.0078  2 2 2 2 2   
               32     1947164    0.4800 ± 0.0078  2 2 2 2 2   
               64     3886108    0.4869 ± 0.0057  2 2 2 2 2   
               128    7763996    0.4820 ± 0.0086  2 2 2 2 2   
               full  15257628    0.5015 ± 0.0052  2 2 2 2 2   

                     last-epoch train acc  last-epoch val acc  \
representation rank                                             
indicator      4                   0.8200              0.3780   
               8                   0.9043              0.4004   
               16                  0.9302              0.4257   
               32                  0.9374              0.4343   
               64                  0.9466              0.4359   
               128                 0.9481              0.4488   
               full                0.9437              0.4897   
bow            4                   0.8218              0.3781   
               8                   0.9028              0.4017   
               16                  0.9303              0.4248   
               32                  0.9378              0.4321   
               64                  0.9462              0.4391   
               128                 0.9472              0.4455   
               full                0.9443              0.4905   
tfidf          4                   0.8039              0.3507   
               8                   0.8778              0.3719   
               16                  0.9118              0.3866   
               32                  0.9252              0.3937   
               64                  0.9395              0.4104   
               128                 0.9454              0.4149   
               full                0.9429              0.4072   

                     last-epoch val loss  
representation rank                       
indicator      4                  3.8360  
               8                  4.1824  
               16                 4.1470  
               32                 4.0445  
               64                 4.1150  
               128                3.9226  
               full               2.9180  
bow            4                  3.7943  
               8                  4.2265  
               16                 4.1559  
               32                 4.0960  
               64                 4.0560  
               128                3.8466  
               full               2.9134  
tfidf          4                  4.0785  
               8                  4.4179  
               16                 4.5387  
               32                 4.5580  
               64                 4.5200  
               128                4.5627  
               full               3.3071